# AIMS Sénégal — Comptage et Suivi du Trafic Routier
**Projet 2 — Vision par Ordinateur | Deadline : 28 Avril 2026**

Ce notebook orchestre le pipeline complet de détection, suivi et comptage d'objets dans des vidéos de trafic.
Toutes les étapes sont appelées via des commandes CLI depuis ce notebook.

## 1. Installation des dépendances

In [ ]:
!pip install -q ultralytics kagglehub filterpy flask pandas plotly seaborn tqdm loguru

## 2. Cloner le projet depuis GitHub

In [ ]:
!git clone https://github.com/ioget/traffic-rookie-aims-cv.git
%cd traffic-rookie-aims-cv

## 3. Téléchargement du dataset BDD100K

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("a7madmostafa/bdd100k-yolo")

print("Path to dataset files:", path)

In [ ]:
# Explorer la structure du dataset
!ls -lh {path}
!find {path} -name '*.yaml' | head -5

In [ ]:
import os

# Exporter le chemin pour les commandes CLI suivantes
DATASET_PATH = path
os.environ['DATASET_PATH'] = DATASET_PATH

# Trouver le fichier yaml de configuration
import glob
yaml_files = glob.glob(f"{DATASET_PATH}/**/*.yaml", recursive=True)
DATASET_YAML = yaml_files[0] if yaml_files else f"{DATASET_PATH}/data.yaml"
os.environ['DATASET_YAML'] = DATASET_YAML
print("Dataset YAML:", DATASET_YAML)

## 4. Vérification de l'environnement (GPU, fichiers)

In [ ]:
!python -c "import torch; print('GPU disponible:', torch.cuda.is_available()); print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
!python -c "from ultralytics import YOLO; print('Ultralytics OK')"
!ls data/
!ls models/

## 5. Fine-tuning du modèle YOLO sur BDD100K

In [ ]:
!python -c "
from src.detection.yolo_detector import YOLODetector
import os
detector = YOLODetector(model_path='models/yolo11n.pt', confidence_threshold=0.5)
detector.fine_tune_model(dataset_config=os.environ['DATASET_YAML'], epochs=10)
print('Fine-tuning terminé.')
"

## 6. Détection + Suivi sur vidéo 1 (traffic.mp4)

In [ ]:
!python main.py \
    --video data/traffic.mp4 \
    --model models/yolo11n.pt \
    --output outputs/traffic_tracked.avi \
    --confidence 0.5 \
    --tracking

## 7. Détection + Suivi sur vidéo 2 (traffic-sign-test.mp4)

In [ ]:
!python main.py \
    --video data/traffic-sign-test.mp4 \
    --model models/yolo11n.pt \
    --output outputs/traffic_sign_tracked.avi \
    --confidence 0.5 \
    --tracking

## 8. Vérification des logs générés

In [ ]:
!ls -lh logs/
!python -c "
import json, glob
log_files = glob.glob('logs/*.json')
for f in log_files:
    with open(f) as fp:
        data = json.load(fp)
    print(f'--- {f} ---')
    if isinstance(data, list):
        print(f'  Total entrées : {len(data)}')
        if data:
            print(f'  Exemple : {data[0]}')
"

## 9. Statistiques agrégées

In [ ]:
!python -c "
import json, glob
from collections import Counter

all_detections = []
for f in glob.glob('logs/*.json'):
    with open(f) as fp:
        data = json.load(fp)
    if isinstance(data, list):
        all_detections.extend(data)

class_counts = Counter(d.get('class_name', 'unknown') for d in all_detections)
print('Comptage total par classe :')
for cls, cnt in class_counts.most_common():
    print(f'  {cls}: {cnt}')
print(f'Total détections : {len(all_detections)}')
"

## 10. Visualisation des résultats

In [ ]:
import matplotlib.pyplot as plt
import json, glob
from collections import Counter

all_detections = []
for f in glob.glob('logs/*.json'):
    with open(f) as fp:
        data = json.load(fp)
    if isinstance(data, list):
        all_detections.extend(data)

class_counts = Counter(d.get('class_name', 'unknown') for d in all_detections)

if class_counts:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(class_counts.keys(), class_counts.values(), color='steelblue')
    ax.set_title('Comptage par classe d\'objet')
    ax.set_xlabel('Classe')
    ax.set_ylabel('Nombre de détections')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig('outputs/stats_classes.png', dpi=150)
    plt.show()
    print('Graphique sauvegardé dans outputs/stats_classes.png')
else:
    print('Aucune donnée de log trouvée. Lance d\'abord les étapes 6 et 7.')

## 11. Lancer l'interface web (optionnel sur Kaggle)

In [ ]:
# Sur Kaggle, l'interface web n'est pas accessible directement
# Ce bloc est prévu pour une exécution locale
# !python main.py --video data/traffic.mp4 --model models/yolo11n.pt --web
print("Interface web : lancer localement avec 'python main.py --video data/traffic.mp4 --web'")

## 12. Vérification des fichiers de sortie

In [ ]:
!ls -lh outputs/ 2>/dev/null || echo 'Dossier outputs/ vide ou inexistant'
!ls -lh logs/ 2>/dev/null || echo 'Dossier logs/ vide ou inexistant'